# VishGuard — Colab demo (T5.2)

Launches the Streamlit UI on Colab T4 and exposes it publicly via an
ngrok tunnel. Use this notebook to record the Phase 5 demo video.

**Run order:** Cell 1 → Cell 1b → Cell 2 → **Cell 2b (pre-warm)** → Cell 3 → Cell 3b

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Add a Colab secret named `NGROK_AUTH_TOKEN` with your free ngrok
   token from [dashboard.ngrok.com](https://dashboard.ngrok.com/authtokens).
3. Set `VISHGUARD_REPO_URL` below (or set it as a Colab secret).

**Demo script (1–2 min):**
- 0:00 — narrate the vishing problem
- 0:15 — show Streamlit UI, upload a WAV
- 0:30 — transcript + spoof score + tactics populate
- 0:50 — risk score card + spoken briefing plays
- 1:10 — mention one known failure (see docs/FAILURES.md)
- 1:25 — show GitHub URL + close

In [ ]:
# Cell 1 — repo setup (self-contained; identical pattern to other notebooks)
import os, sys, subprocess, shutil
from pathlib import Path

REPO_URL = os.environ.get('VISHGUARD_REPO_URL', 'https://github.com/PLACEHOLDER/vishguard.git')
DRIVE_SRC = '/content/drive/MyDrive/vishguard'
REPO_DIR  = Path('/content/vishguard')
ON_COLAB  = 'google.colab' in sys.modules or Path('/content').exists()

def sh(cmd, check=True):
    print('$', cmd)
    subprocess.run(cmd, shell=True, check=check)

if ON_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except Exception as e:
        print('Drive mount skipped:', e)

    if 'PLACEHOLDER' not in REPO_URL:
        if REPO_DIR.exists():
            sh(f'cd {REPO_DIR} && git pull --ff-only')
        else:
            sh(f'git clone {REPO_URL} {REPO_DIR}')
    elif Path(DRIVE_SRC).exists():
        REPO_DIR.mkdir(parents=True, exist_ok=True)
        sh(f'rsync -a --delete --exclude .venv --exclude __pycache__ {DRIVE_SRC}/ {REPO_DIR}/')
    else:
        raise RuntimeError('Set VISHGUARD_REPO_URL or copy the repo to MyDrive/vishguard/')

    sh(f'pip install -q -e {REPO_DIR}')
    sh(f'pip install -q -r {REPO_DIR}/requirements.txt')
    try:
        import torch
        if torch.cuda.is_available():
            sh(f'pip install -q -r {REPO_DIR}/requirements-gpu.txt')
    except Exception as e:
        print('GPU extras skipped:', e)

    src = str(REPO_DIR / 'src')
    if src not in sys.path:
        sys.path.insert(0, src)

import vishguard
print('vishguard', vishguard.__version__)

## Cell 1b — Generate demo audio (optional)

Run this cell to synthesise a realistic vishing call WAV using SpeechT5.
Skip if you already have `demo_call.wav` uploaded to `/tmp/demo_call.wav`.

In [ ]:
# Cell 1b — generate a synthetic vishing call WAV using SpeechT5
# Speaker embedding approach mirrors notebooks/01_spike_antiSpoof.ipynb cell-4.
import pathlib, zipfile, numpy as np, torch, soundfile as sf
from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan
from huggingface_hub import snapshot_download

SEGMENTS = [
    ('Hello. This is officer Daniel Reed from the Social Security '
     'Administration fraud division. We have detected suspicious activity '
     'linked to your social security number, which has now been suspended.'),
    ('You are facing a penalty of twenty four hundred dollars due to '
     'fraudulent use of your account. To avoid immediate arrest and a '
     'federal warrant being issued in your name, you must act now.'),
    ('Call our resolution hotline and provide your social security number '
     'and bank account details to verify your identity. '
     'You have twenty four hours before legal proceedings begin. '
     'This is your final warning.'),
]

DEMO_WAV    = pathlib.Path('/tmp/demo_call.wav')
SAMPLE_RATE = 16_000
device      = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

print('Loading SpeechT5 models...')
processor = SpeechT5Processor.from_pretrained('microsoft/speecht5_tts')
model     = SpeechT5ForTextToSpeech.from_pretrained('microsoft/speecht5_tts').to(device)
vocoder   = SpeechT5HifiGan.from_pretrained('microsoft/speecht5_hifigan').to(device)

print('Loading speaker embedding...')
xvec_repo = snapshot_download(
    repo_id='Matthijs/cmu-arctic-xvectors',
    repo_type='dataset',
    ignore_patterns=['*.py'],
)
zip_path    = pathlib.Path(xvec_repo) / 'spkrec-xvect.zip'
extract_dir = pathlib.Path('/tmp/cmu_arctic_xvect')
extract_dir.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(extract_dir)

npy_files = sorted(extract_dir.rglob('*.npy'))
pt_files  = sorted(extract_dir.rglob('*.pt'))

if npy_files:
    xvec    = np.load(npy_files[7306 % len(npy_files)])
    speaker = torch.tensor(xvec).unsqueeze(0).to(device)
elif pt_files:
    data = torch.load(pt_files[0], map_location='cpu')
    if isinstance(data, torch.Tensor):
        speaker = data[min(7306, data.shape[0] - 1)].unsqueeze(0).to(device)
    elif isinstance(data, dict):
        speaker = torch.stack(list(data.values()))[0].unsqueeze(0).to(device)
    else:
        raise ValueError(f'unexpected pt type: {type(data)}')
else:
    print('WARNING: using normalized random embedding')
    speaker = torch.nn.functional.normalize(torch.randn(1, 512), dim=-1).to(device)

print(f'speaker shape: {speaker.shape}')

print('Synthesising segments...')
silence = np.zeros(int(SAMPLE_RATE * 0.35), dtype=np.float32)
parts   = []
for i, text in enumerate(SEGMENTS, 1):
    print(f'  {i}/{len(SEGMENTS)}: {text[:60]}...')
    inputs = processor(text=text, return_tensors='pt').to(device)
    with torch.no_grad():
        speech = model.generate_speech(
            inputs['input_ids'], speaker, vocoder=vocoder
        )
    parts.append(speech.cpu().numpy())
    parts.append(silence)

combined = np.concatenate(parts)
sf.write(str(DEMO_WAV), combined, samplerate=SAMPLE_RATE)
print(f'\nSaved: {DEMO_WAV}  ({len(combined)/SAMPLE_RATE:.1f}s)')

from google.colab import files
files.download(str(DEMO_WAV))

In [ ]:
# Cell 2 — install pyngrok and configure auth token
import subprocess
subprocess.run(['pip', 'install', '-q', 'pyngrok'], check=True)

import os
from pyngrok import ngrok, conf

try:
    from google.colab import userdata
    ngrok_token = userdata.get('NGROK_AUTH_TOKEN')
except Exception:
    ngrok_token = os.environ.get('NGROK_AUTH_TOKEN', '')

if not ngrok_token:
    raise RuntimeError(
        'Set NGROK_AUTH_TOKEN in Colab secrets (Secrets panel on the left) '
        'or as an environment variable. Free account at dashboard.ngrok.com.'
    )

conf.get_default().auth_token = ngrok_token
print('ngrok auth token configured')

## Cell 2b — Pre-warm all models (run BEFORE Cell 3)

This cell runs the full pipeline via the CLI on the demo audio.
It downloads and caches all model weights to disk (~3–5 min on a fresh
runtime) so that when Streamlit runs the same analysis it only has to
load from disk into GPU — reducing in-app wait time to ~30–60 s.

**Do not skip this cell.** Without it, Streamlit downloads weights on
the first click and the spinner will run for 5+ minutes.

In [ ]:
# Cell 2b — CLI pre-warm: downloads + caches all model weights before Streamlit starts
import subprocess, json, time
from pathlib import Path

REPO_DIR = Path('/content/vishguard')
AUDIO    = Path('/tmp/demo_call.wav')
OUT_DIR  = Path('/tmp/vishguard_prewarm')
OUT_DIR.mkdir(exist_ok=True)

if not AUDIO.exists():
    raise FileNotFoundError(
        '/tmp/demo_call.wav not found — run Cell 1b first, '
        'or upload the WAV from artifacts/audio/demo_call.wav'
    )

print('Running full pipeline on demo audio (downloads model weights)...')
print('Expected: 3–5 min on a fresh runtime, ~60 s if weights already cached.')
t0 = time.time()

result = subprocess.run(
    [
        'python', '-m', 'vishguard.cli',
        'run',
        str(AUDIO),
        '--out',    str(OUT_DIR),
        '--device', 'cuda',
        '--prompt', 'v4',
        '--no-tts',
    ],
    cwd=str(REPO_DIR),
    capture_output=True,
    text=True,
)

elapsed = time.time() - t0
print(f'Done in {elapsed:.0f}s')
print(result.stdout)

if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])
    raise RuntimeError('Pre-warm failed — fix the error above before starting Streamlit.')

print('All weights cached. Streamlit first-run will be ~30–60 s instead of 5+ min.')

In [ ]:
# Cell 3 — launch Streamlit in background, open ngrok tunnel
import subprocess, time
from pathlib import Path
from pyngrok import ngrok

REPO_DIR = Path('/content/vishguard')
PORT     = 8501
LOG_FILE = '/tmp/streamlit.log'

_log_fh = open(LOG_FILE, 'w')
_streamlit_proc = subprocess.Popen(
    [
        'streamlit', 'run',
        str(REPO_DIR / 'app.py'),
        f'--server.port={PORT}',
        '--server.headless=true',
        '--server.enableCORS=false',
        '--server.enableXsrfProtection=false',
    ],
    cwd=str(REPO_DIR),
    stdout=_log_fh,
    stderr=_log_fh,
)

time.sleep(5)

tunnel     = ngrok.connect(PORT, 'http')
public_url = tunnel.public_url

print('=' * 60)
print('VishGuard Streamlit UI:')
print(public_url)
print('=' * 60)
print(f'Logs: {LOG_FILE}  (run Cell 3b to check them)')

In [ ]:
# Cell 3b — check GPU memory + tail Streamlit log
# Run any time to verify CUDA is in use and see pipeline progress.
import subprocess

print('=== GPU memory ===')
r = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.used,memory.free,utilization.gpu',
     '--format=csv,noheader'],
    capture_output=True, text=True
)
print(r.stdout.strip() or 'nvidia-smi not available')

print('\n=== Streamlit log (last 60 lines) ===')
try:
    lines = open('/tmp/streamlit.log').readlines()
    print(''.join(lines[-60:]))
except FileNotFoundError:
    print('Log not found — run Cell 3 first.')

In [ ]:
# Cell 4 — shutdown (run after recording to free port + kill tunnel)
from pyngrok import ngrok

ngrok.kill()

try:
    _streamlit_proc.terminate()
    _streamlit_proc.wait(timeout=5)
    _log_fh.close()
    print('Streamlit stopped')
except Exception as e:
    print('Shutdown:', e)